# W1 — Mask2Former on Fashionpedia, judged by measured volume

This notebook fine-tunes Mask2Former (Swin-S, COCO instance checkpoint) on
Fashionpedia's 14 whole-garment classes plus `sleeve`, then does the part that
decides whether W1 worked: it measures `volume` from masks on **your own
photos** and compares it with the hand-judged column in `data/wardrobe.csv`.

Mask mIoU is reported, but it is not the success criterion. The target is
measured volume within **±0.05** of the engine's scale (see
`docs/w1-segmentation.md` for why that number, and what "the engine's scale"
means).

**Runtime:** a GPU. On a 16 GB card (T4, L4) keep `batch_size=4`; on 40 GB+
(A100) use 8. Training checkpoints every 2,000 steps to `OUT` and resumes from
the last checkpoint if the session drops, so put `OUT` on Google Drive.

**Licence:** Fashionpedia's annotations are CC BY 4.0. Each image keeps its own
licence, and most are NonCommercial. `LICENCE = "commercial"` trains only on
images whose licence allows commercial use (about 30% of them).

In [ ]:
# 1. Code. The branch has the W1 package; the extra installs torch-side deps.
!git clone -b w1-segmentation https://github.com/PujaSridhar/silhouette.git
%cd silhouette
!pip install -q -e '.[vision]'

In [ ]:
# Optional: keep data and checkpoints on Drive so a disconnect loses nothing.
# from google.colab import drive; drive.mount('/content/drive')
DATA = "data/fashionpedia"          # or "/content/drive/MyDrive/fashionpedia"
OUT = "runs/w1"                     # or "/content/drive/MyDrive/silhouette/runs/w1"
LICENCE = "all"                     # "commercial" for a model that might ship
BATCH = 4                           # 8 on an A100

## 2. Fashionpedia

About 3.6 GB of images and 0.6 GB of annotations. The validation images unzip
into a folder called `test/` — that is Fashionpedia's naming, not a mistake.

In [ ]:
import os
os.makedirs(DATA, exist_ok=True)
base = "https://s3.amazonaws.com/ifashionist-dataset"
for f in ["annotations/instances_attributes_train2020.json",
          "annotations/instances_attributes_val2020.json",
          "images/train2020.zip", "images/val_test2020.zip"]:
    dest = os.path.join(DATA, os.path.basename(f))
    if not os.path.exists(dest):
        !wget -q -O {dest} {base}/{f}
for z in ["train2020.zip", "val_test2020.zip"]:
    !cd {DATA} && unzip -q -n {z}
!ls {DATA}

In [ ]:
!silhouette-vision fashionpedia {DATA}/instances_attributes_train2020.json --licence {LICENCE}

## 3. A short run first

Two hundred steps on 500 images. The loss should fall clearly (it starts in
the 20s-40s). If it does not, stop here rather than spending the long run.

In [ ]:
from silhouette.vision.train import TrainConfig, fit, evaluate
from silhouette.vision.fashionpedia import load

train = load(f"{DATA}/instances_attributes_train2020.json", licence=LICENCE, min_area=64)
val = load(f"{DATA}/instances_attributes_val2020.json", licence="all")
print(len(train), "train images,", len(val), "val images")

quick = TrainConfig(images=f"{DATA}/train", out=f"{OUT}-quick", batch_size=BATCH,
                    max_steps=200, warmup_steps=50, subset=500, save_every=200, eval_every=0)
print(fit(quick, samples=train))

## 4. The real run

`max_steps=30000` at batch 4 is about 2.6 epochs. Re-running this cell after a
disconnect resumes from `OUT/last.pt`.

In [ ]:
cfg = TrainConfig(images=f"{DATA}/train", val_images=f"{DATA}/test", out=OUT,
                  licence=LICENCE, batch_size=BATCH, max_steps=30000,
                  eval_every=5000, eval_limit=300)
print(fit(cfg, samples=train, val_samples=val))

## 5. Mask accuracy on Fashionpedia val

Per-class IoU and IoU pooled by engine category (top, outer, bottom, dress,
shoe). This is the conventional number. Read it, then move on: it does not say
whether the mask edges are good enough to measure width from.

In [ ]:
import json
from transformers import Mask2FormerForUniversalSegmentation
model = Mask2FormerForUniversalSegmentation.from_pretrained(f"{OUT}/model").to("cuda")
res = evaluate(model, val, f"{DATA}/test")
json.dump(res, open(f"{OUT}/val_iou.json", "w"), indent=2)
print("mIoU", round(res["mIoU"], 3))
for k, v in res["by_engine_category"].items():
    print(f"  {k:<8} {v if v is None else round(v, 3)}")

## 6. Your photos: the number that decides W1

Upload your `photos/` folder (see `docs/w1-segmentation.md` for how to take
them and what goes in `manifest.csv`). It should already contain your hand
masks from the SAM 2 helper in `photos/masks/`.

The cells below:
1. run the model on every photo and write `photos/masks_model/`,
2. measure volume from your hand masks and from the model's masks,
3. show how many outfits would change which rules fire if the measured
   (calibrated) volumes replaced the judged ones.

Photos stay out of git; `photos/` is gitignored.

In [ ]:
from silhouette.models import load_wardrobe
from silhouette.vision.train import predict_photos
for note in predict_photos(model, "photos", load_wardrobe("data/wardrobe.csv")):
    print(note)

In [ ]:
# Model masks against your hand masks: the phone-photo check.
!silhouette-vision compare-masks --photos photos
!silhouette-vision measure --photos photos --masks masks       --out evals/volume/hand.csv
!silhouette-vision measure --photos photos --masks masks_model --out evals/volume/model.csv

In [ ]:
# Calibrated measured volumes -> CSV -> how much of the engine's advice changes.
import csv, json
s = json.load(open("evals/volume/model.json"))["default"]["per_garment"]
rows = {}
for key, v in s.items():
    gid, field = key.split(":")
    if v["calibrated"] is not None:
        rows.setdefault(gid, {})[field] = v["calibrated"]
with open("evals/volume/model_calibrated.csv", "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["garment_id", "volume", "volume_lower"])
    for gid, d in rows.items():
        w.writerow([gid, d.get("volume", ""), d.get("volume_lower", "")])
!silhouette-vision sensitivity --volumes evals/volume/model_calibrated.csv

## 7. Keep the model

Copy `OUT/model` somewhere durable (Drive, or a Hugging Face model repo). It
is about 270 MB. Do not commit it to the silhouette repo.